<a href="https://colab.research.google.com/github/sreekanthTa/BreedPredictionDL/blob/main/rain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [69]:
import pandas as pd

In [70]:
data = pd.read_csv('/content/drive/MyDrive/DeepLearning/Rain/rain.csv')
data.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [71]:
data["Date"] = pd.to_datetime(data["Date"])

data["Year"] = data["Date"].dt.year
data["Month"] = data["Date"].dt.month
data["Day"] = data["Date"].dt.day

In [72]:
data["RainToday"] = data["RainToday"].str.strip().str.capitalize()

data["RainToday"] = data["RainToday"].map({'Yes': 1, 'No': 0})

data["RainToday"].value_counts()

data["RainTomorrow"] = data["RainTomorrow"].str.strip().str.capitalize()

data["RainTomorrow"] = data["RainTomorrow"].map({'Yes':1, 'No':0})

In [73]:
X = data.drop("RainTomorrow", axis=1)
Y = data["RainTomorrow"]

X.shape, Y.shape


((145460, 25), (145460,))

In [75]:
categorical_features = ['WindGustDir','WindDir9am','WindDir3pm','Location']

dummies = pd.get_dummies(
    data=X[categorical_features],
    dtype=float
)

# dummies.columns

# Drop original categorical columns
X = X.drop(columns=categorical_features)

# Concatenate dummies with the original Xset
X = pd.concat([X, dummies], axis=1)



In [79]:


# // Skewed Ho for Rainfall, Evaporation, WindsurSpeed, WindSpeed9am, windSpeed3pm, Humiidyt9pam
skewed_columns = ['Rainfall', 'Evaporation','WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm','Humidity9am']

for col in skewed_columns:
    X[col] = X[col].fillna(X[col].median())

# //Normal Ho for MinTemo, Ho for MaxTemp, Ho for sumshine,
normal_columns = ['MinTemp', 'MaxTemp','Sunshine', 'Humidity3pm','Pressure9am','Pressure3pm', 'Temp9am','Temp3pm','Cloud9am','Cloud3pm']
# WindGustDir, WindDir9am, WindDir3pm

for col in normal_columns:
   X[col] = X[col].fillna(X[col].mean())


# text_columns = ['WindGustDir','WindDir9am', 'WindDir3pm']
# for col in text_columns:
#    X[col] = X[col].fillna('missing')


yes_no = ['RainToday']
for col in yes_no:
   X[col] = X[col].fillna(X[col].mode()[0])


In [80]:
print(X.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Columns: 118 entries, Date to Location_Woomera
dtypes: datetime64[ns](1), float64(114), int32(3)
memory usage: 129.3 MB
None


In [ ]:
# X = data.drop(columns=['RainTomorrow'], axis=1)
# y = data['RainTomorrow']

In [81]:
from sklearn.model_selection import train_test_split
X_train, X_test,  y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((116368, 118), (29092, 118), (116368,), (29092,))

In [82]:
import tensorflow as tf;
print(tf.__version__)

2.18.0


In [83]:
X_train['Date'] = pd.to_numeric(X_train['Date'])  # Convert Date to Unix timestamp
X_test['Date'] = pd.to_numeric(X_test['Date'])    # Do the same for the test set


In [84]:
X_train.describe()

,Date,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,...,Location_Townsville,Location_Tuggeranong,Location_Uluru,Location_WaggaWagga,Location_Walpole,Location_Watsonia,Location_Williamtown,Location_Witchcliffe,Location_Wollongong,Location_Woomera
count,1.163680e+05,116368.000000,116368.000000,116368.000000,116368.000000,116368.000000,116368.000000,116368.000000,116368.000000,116368.000000,...,116368.000000,116368.00000,116368.000000,116368.000000,116368.000000,116368.000000,116368.000000,116368.000000,116368.000000,116368.000000
mean,1.365292e+18,12.197050,23.219558,2.325638,5.177513,7.612113,39.963031,14.039427,18.670597,68.927927,...,0.020865,0.02102,0.010802,0.020564,0.020538,0.020504,0.020538,0.020762,0.020882,0.020547
std,7.965361e+16,6.367265,7.086948,8.448650,3.177739,2.727652,13.144589,8.865300,8.725620,18.842219,...,0.142932,0.14345,0.103370,0.141920,0.141833,0.141717,0.141833,0.142586,0.142990,0.141862
min,1.193962e+18,-8.500000,-4.800000,0.000000,0.000000,0.000000,6.000000,0.000000,0.000000,0.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.294704e+18,7.700000,18.000000,0.000000,4.200000,7.611178,31.000000,7.000000,13.000000,57.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.370390e+18,12.100000,22.700000,0.000000,4.800000,7.611178,39.000000,13.000000,19.000000,70.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.434586e+18,16.800000,28.200000,0.600000,5.200000,8.700000,46.000000,19.000000,24.000000,83.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.498349e+18,31.900000,47.300000,371.000000,145.000000,14.500000,135.000000,130.000000,87.000000,100.000000,...,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
model_1 = tf.keras.Sequential([
    tf.keras.layers.Dense(5, input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(5, activation="relu"),
    tf.keras.layers.Dense(5, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")

])

model_1.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"]
)

history_1 = model_1.fit(X_train, y_train, epochs=5)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
3637/3637 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.7580 - loss: nan
Epoch 2/5
3637/3637 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.7595 - loss: nan
Epoch 3/5
3637/3637 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.7605 - loss: nan
Epoch 4/5
1017/3637 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.7643 - loss: nan

In [ ]:
pd.DataFrame(history_1.history).plot()